### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### 1. 
Tomamos 5 documentos al azar del conjunto de entrenamiento y calculamos la similaridad coseno contra el resto del corpus. Luego, revisamos las clases de los 5 documentos más similares para ver si tienen coherencia con la etiqueta original.

In [26]:
np.random.seed(1234)
random_indices = np.random.choice(X_train.shape[0], 5, replace=False)

for idx in random_indices:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    
    mostsim = np.argsort(cossim)[::-1][1:6]
    
    clase_original = newsgroups_train.target_names[y_train[idx]]
    print(f"Documento ID {idx} - Clase original: {clase_original}")
    print("Clases de los 5 similares:")
    
    for i in mostsim:
        clase_sim = newsgroups_train.target_names[y_train[i]]
        print(f"{clase_sim} (similitud coseno: {cossim[i]:.3f})")
    print("\n")

Documento ID 545 - Clase original: sci.space
Clases de los 5 similares:
sci.space (similitud coseno: 0.356)
sci.space (similitud coseno: 0.352)
sci.space (similitud coseno: 0.335)
sci.space (similitud coseno: 0.327)
sci.space (similitud coseno: 0.316)


Documento ID 7251 - Clase original: comp.windows.x
Clases de los 5 similares:
talk.politics.guns (similitud coseno: 0.274)
sci.crypt (similitud coseno: 0.191)
comp.windows.x (similitud coseno: 0.182)
comp.os.ms-windows.misc (similitud coseno: 0.180)
sci.crypt (similitud coseno: 0.171)


Documento ID 4045 - Clase original: talk.politics.mideast
Clases de los 5 similares:
talk.politics.mideast (similitud coseno: 0.408)
talk.politics.mideast (similitud coseno: 0.396)
talk.politics.mideast (similitud coseno: 0.369)
talk.politics.mideast (similitud coseno: 0.362)
talk.politics.mideast (similitud coseno: 0.340)


Documento ID 10160 - Clase original: rec.sport.hockey
Clases de los 5 similares:
comp.graphics (similitud coseno: 0.212)
rec.sport.

Vemos que la similitud coseno funciona bastante bien para agrupar temas muy marcados. En los casos del espacio, medio oriente y criptografía, los 5 documentos más cercanos pertenecen exactamente a la misma categoría y con similitudes mayores a 0.30 o 0.35.

Por otro lado, hay temáticas en las que se aleja por completo: en el documento de Windows X terminó trayendo foros de armas, y en el de hockey trajo de gráficos o motos. Lo interesante es que en estos casos de error, los valores de similitud son bastante más bajos (entre 0.16 y 0.27). Probablemente esos documentos eran cortos, tenían mucho ruido o palabras genéricas que confundieron al vectorizador.

### 2. 
Para este paso, vamos a predecir la etiqueta de cada documento de test buscándole su equivalente más parecido en el conjunto de entrenamiento mediante similitud coseno.

In [28]:
# clasificación por prototipos 
sim_matrix = cosine_similarity(X_test, X_train)

# Para cada fila (doc de test), buscamos el índice de la columna (doc de train) con mayor similitud
best_matches_idx = sim_matrix.argmax(axis=1)

# Asignamos la clase de ese documento de train como predicción
y_pred_prototipos = y_train[best_matches_idx]

# Evaluamos
f1_prototipos = f1_score(y_test, y_pred_prototipos, average='macro')
print(f"F1-Score: {f1_prototipos:.4f}")

F1-Score: 0.5050


Vemos que el modelo por prototipos dio 0.5050, que es bastante más bajo que el 0.5854 que nos había dado el Naive Bayes Multinomial original. Esto tiene sentido porque el método de prototipos depende de encontrar un texto casi idéntico y es muy sensible al ruido de cada documento individual. Un modelo probabilístico como Naive Bayes generaliza mejor porque aprende de las frecuencias de las palabras en toda la clase junta, no en un solo documento.


### 3.
Ahora vamos a intentar superar ese 0.58. Como no podemos tocar el `ngram_range`, vamos a mejorar el vectorizador filtrando stopwords y palabras muy raras/comunes (palabras que aparecenen más del 50% de los textos o en menos de 3 textos). Después probamos el `MultinomialNB` y el `ComplementNB` con un par de valores de `alpha` para ver cuál rinde más.

In [31]:
tfidfvect_mejorado = TfidfVectorizer(stop_words='english', max_df=0.5, min_df=3)

X_train_mejorado = tfidfvect_mejorado.fit_transform(newsgroups_train.data)
X_test_mejorado = tfidfvect_mejorado.transform(newsgroups_test.data)

# probamos ambos modelos con un par de alphas
modelos = [MultinomialNB, ComplementNB]
alphas = [0.1, 0.5, 1.0]

for Modelo in modelos:
    print(f"{Modelo.__name__}")
    for a in alphas:
        clf = Modelo(alpha=a)
        clf.fit(X_train_mejorado, y_train)
        y_pred = clf.predict(X_test_mejorado)
        f1 = f1_score(y_test, y_pred, average='macro')
        print(f"Alpha {a}: F1-Macro = {f1:.4f}")
    print("\n")

MultinomialNB
Alpha 0.1: F1-Macro = 0.6818
Alpha 0.5: F1-Macro = 0.6642
Alpha 1.0: F1-Macro = 0.6530


ComplementNB
Alpha 0.1: F1-Macro = 0.6822
Alpha 0.5: F1-Macro = 0.6921
Alpha 1.0: F1-Macro = 0.6916




Logramos saltar del 0.58 original a un 0.6921 usando `ComplementNB` con `alpha=0.5`. 
Filtrar las stop words y las palabras muy raras o muy comunes limpia bastante el ruido. Resulta claro que el modelo Complement Naive Bayes es mejor para este tipo de datos de texto que el Multinomial tradicional.


### 4. 
Ahora vamos a dar trasponer la matriz para que las filas sean las palabras y las columnas los documentos. Con esto podemos buscar qué palabras se usan en contextos similares. Elegí 5 palabras: **dios (god), nave (ship), casa (house), bate (bat) y juego (game)** y veamos si los resultados tienen sentido.

In [33]:
# Transponemos la matriz (usamos la que no tiene stopwords)
X_train_T = X_train_mejorado.T

# Diccionarios para ir de palabra a índice y viceversa
vocab = tfidfvect_mejorado.vocabulary_
idx2word_mejorado = {v: k for k, v in vocab.items()}

# Palabras
palabras_manuales = ['god', 'ship', 'house', 'bat', 'game']

for palabra in palabras_manuales:
    
    idx = vocab[palabra]
    
    cossim_word = cosine_similarity(X_train_T[idx], X_train_T)[0]
    
    # Tomamos los índices de las 5 palabras más parecidas (salteando la pos 0 que es la palabra misma)
    mostsim_words_idx = np.argsort(cossim_word)[::-1][1:6]
    
    print(f"Palabra elegida: {palabra.upper()}")
    print("5 palabras más similares:")
    for i in mostsim_words_idx:
        print(f"{idx2word_mejorado[i]} (similitud coseno: {cossim_word[i]:.3f})")
    print("\n")

Palabra elegida: GOD
5 palabras más similares:
jesus (similitud coseno: 0.277)
bible (similitud coseno: 0.273)
christ (similitud coseno: 0.267)
faith (similitud coseno: 0.258)
existence (similitud coseno: 0.253)


Palabra elegida: SHIP
5 palabras más similares:
prep (similitud coseno: 0.208)
589 (similitud coseno: 0.187)
clemente (similitud coseno: 0.181)
evaporated (similitud coseno: 0.169)
controllers (similitud coseno: 0.163)


Palabra elegida: HOUSE
5 palabras más similares:
senate (similitud coseno: 0.266)
white (similitud coseno: 0.240)
cpr (similitud coseno: 0.231)
veto (similitud coseno: 0.212)
deposit (similitud coseno: 0.203)


Palabra elegida: BAT
5 palabras más similares:
autoexec (similitud coseno: 0.480)
config (similitud coseno: 0.253)
sys (similitud coseno: 0.236)
pucks (similitud coseno: 0.184)
gloved (similitud coseno: 0.152)


Palabra elegida: GAME
5 palabras más similares:
games (similitud coseno: 0.213)
espn (similitud coseno: 0.187)
hockey (similitud coseno: 0.182

Interpretación de los resultados de cada palabra:

* **GOD y GAME:** Funcionan muy bien y arman clústeres obvios. Agrupan términos religiosos (jesus, bible, faith) y deportivos (espn, hockey, scored) de forma casi perfecta.
* **HOUSE:** El modelo no la asoció a "hogar", sino a su uso político ("The White House"), por eso trae términos como senate y veto.
* **BAT:** "Bat" (Bate) no se asoció al deporte de béisbol como pretendí, sino a los archivos de sistema de MS-DOS (autoexec.bat, config.sys). Recién al final aparecen términos deportivos (gloved, pucks).
* **SHIP:** Trajo bastante ruido y cosas mezcladas (prep, 589). Al parecer la palabra se usó en contextos demasiado variados y su vector quedó diluido.

### **CONCLUSIONES GENERALES**

El desarrollo de este primer desafío deja en claro que el preprocesamiento del texto es crítico. Filtrar el ruido al eliminar stopwords y ajustar los umbrales de frecuencia permitió dar un salto significativo en la métrica de evaluación. 

Asimismo, se comprobó que los enfoques probabilísticos, en especial la variante `ComplementNB`, suele trabajar mejor con el desbalance de clases y superó ampliamente a los métodos de clasificación basados únicamente en la similitud de prototipos. 

Por último, el análisis de la matriz transpuesta evidenció cómo los vectores absorben el contexto histórico y temático particular del corpus (como la fuerte asociación de "bat" con MS-DOS o de "house" con la política estadounidense), demostrando que siempre es necesario explorar y comprender los datos  antes de meterse en las métricas de un modelo.